# Day 5: Deep & Robust CNNs
## Regularization, Batch Normalization, and Optimization
---
**Objective:** Transition from a simple model to a production-grade architecture using FashionMNIST.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

Executing on: cuda


### 5.1 Advanced Data Augmentation
We use `RandomAffine` and `ColorJitter` to ensure the model learns features, not positions.

In [2]:
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomAffine(0, translate=(0.1, 0.1)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

test_transform = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])

train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)

### 5.2 Modular Architecture with Kaiming Initialization
For ReLU networks, we use **Kaiming (He) Initialization** to prevent vanishing gradients in the first epoch.

In [3]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        
        # Initialize weights
        nn.init.kaiming_normal_(self.conv.weight, nonlinearity='relu')
        
    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))

class DeepFashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1, 32), # 28x28 -> 14x14
            ConvBlock(32, 64) # 14x14 -> 7x7
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        return self.fc(self.encoder(x))

model = DeepFashionCNN().to(device)
print(model)

DeepFashionCNN(
  (encoder): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU()
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


### 5.3 Learning Rate Scheduler
We use `StepLR` to reduce the learning rate by half every 5 epochs to fine-tune the weights as we approach the minimum.

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Optimizer and Scheduler initialized.")

Optimizer and Scheduler initialized.


### 5.4 Professional Training Loop with Validation
Tracking both Train and Test accuracy in a single loop.

In [5]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

Epoch 1/10 | Loss: 0.7281 | LR: 0.001000
Epoch 2/10 | Loss: 0.5364 | LR: 0.001000
Epoch 3/10 | Loss: 0.4810 | LR: 0.001000
Epoch 4/10 | Loss: 0.4430 | LR: 0.001000
Epoch 5/10 | Loss: 0.4195 | LR: 0.000500
Epoch 6/10 | Loss: 0.3825 | LR: 0.000500
Epoch 7/10 | Loss: 0.3700 | LR: 0.000500
Epoch 8/10 | Loss: 0.3583 | LR: 0.000500
Epoch 9/10 | Loss: 0.3548 | LR: 0.000500
Epoch 10/10 | Loss: 0.3492 | LR: 0.000250
